я надумав опрацювати краще дані по вінрейту кожної з команд - знайти standart deviation між перемогами вдома і гостях, 1. outliers - команди які краще грають вдома краще, ніж інші. тобто в який відсоток переміг замітно більший за sd. 2. також список команд, які грають краще на виїзді 3. відсортувати по сезонам і по лігам цю інформацію

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import sqlite3
import helpers as hp
conn = sqlite3.connect('database.sqlite')

In [2]:
#create df with data needed
matches_df = pd.read_sql("SELECT match_api_id, league_id, season, home_team_api_id, away_team_api_id, home_team_goal, away_team_goal FROM match", conn)  

In [3]:
#add column with home team name
matches_df = hp.team_names(matches_df, "home_team_api_id", "home_team")

In [4]:
#add column with away team name
matches_df = hp.team_names(matches_df,"away_team_api_id", "away_team")

In [5]:
matches_df = hp.league_names(matches_df)

In [6]:
#add column with result
matches_df = hp.result(matches_df)

In [7]:
matches_df.head()

,match_api_id,season,home_team_api_id,away_team_api_id,home_team_goal,away_team_goal,home_team,away_team,League,result
0,492473,2008/2009,9987,9993,1,1,KRC Genk,Beerschot AC,Belgium Jupiler League,draw
1,492474,2008/2009,10000,9994,0,0,SV Zulte-Waregem,Sporting Lokeren,Belgium Jupiler League,draw
2,492475,2008/2009,9984,8635,0,3,KSV Cercle Brugge,RSC Anderlecht,Belgium Jupiler League,away_win
3,492476,2008/2009,9991,9998,5,0,KAA Gent,RAEC Mons,Belgium Jupiler League,home_win
4,492477,2008/2009,7947,9985,1,3,FCV Dender EH,Standard de Liège,Belgium Jupiler League,away_win


In [8]:
team_match_home = matches_df[[
    'home_team_api_id',
    'home_team',
    'League',
    'season', 
    'home_team_goal',
    'away_team_goal',
    'result'
    ]].rename(
    columns = {
        'home_team_api_id':"team_id",
        'home_team': "Club",
        'season': "Season",
        'home_team_goal': "GF",
        'away_team_goal': "GA"}
    )

In [9]:
team_match_home.head()

,team_id,Club,League,Season,GF,GA,result
0,9987,KRC Genk,Belgium Jupiler League,2008/2009,1,1,draw
1,10000,SV Zulte-Waregem,Belgium Jupiler League,2008/2009,0,0,draw
2,9984,KSV Cercle Brugge,Belgium Jupiler League,2008/2009,0,3,away_win
3,9991,KAA Gent,Belgium Jupiler League,2008/2009,5,0,home_win
4,7947,FCV Dender EH,Belgium Jupiler League,2008/2009,1,3,away_win


In [10]:
#creating dataframe for further analysis
team_match_away = matches_df[[
    'away_team_api_id',
    'away_team',
    'League',
    'season', 
    'away_team_goal',
    'home_team_goal',
    'result'
    ]].rename(
    columns = {
        'away_team_api_id':"team_id",
        'away_team': "Club",
        'season': "Season",
        'away_team_goal': "GF",
        'home_team_goal': "GA"}
    )

In [11]:
team_match_away.head()

,team_id,Club,League,Season,GF,GA,result
0,9993,Beerschot AC,Belgium Jupiler League,2008/2009,1,1,draw
1,9994,Sporting Lokeren,Belgium Jupiler League,2008/2009,0,0,draw
2,8635,RSC Anderlecht,Belgium Jupiler League,2008/2009,3,0,away_win
3,9998,RAEC Mons,Belgium Jupiler League,2008/2009,0,5,home_win
4,9985,Standard de Liège,Belgium Jupiler League,2008/2009,3,1,away_win


In [12]:
#changing the result column data to actual
team_match_home["result"] = team_match_home["result"].replace({
    "home_win":"W",
    "away_win":"L",
    "draw":"D"})

In [13]:
team_match_home.head()

,team_id,Club,League,Season,GF,GA,result
0,9987,KRC Genk,Belgium Jupiler League,2008/2009,1,1,D
1,10000,SV Zulte-Waregem,Belgium Jupiler League,2008/2009,0,0,D
2,9984,KSV Cercle Brugge,Belgium Jupiler League,2008/2009,0,3,L
3,9991,KAA Gent,Belgium Jupiler League,2008/2009,5,0,W
4,7947,FCV Dender EH,Belgium Jupiler League,2008/2009,1,3,L


In [14]:
team_match_away["result"] = team_match_away["result"].replace({
    "home_win":"L",
    "away_win":"W",
    "draw":"D"})

In [15]:
team_match_home.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25979 entries, 0 to 25978
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   team_id  25979 non-null  int64 
 1   Club     25979 non-null  object
 2   League   25979 non-null  object
 3   Season   25979 non-null  object
 4   GF       25979 non-null  int64 
 5   GA       25979 non-null  int64 
 6   result   25979 non-null  object
dtypes: int64(3), object(4)
memory usage: 1.4+ MB


In [16]:
match_results = pd.concat([team_match_away,team_match_home], ignore_index = True)

In [17]:
match_results.head()

,team_id,Club,League,Season,GF,GA,result
0,9993,Beerschot AC,Belgium Jupiler League,2008/2009,1,1,D
1,9994,Sporting Lokeren,Belgium Jupiler League,2008/2009,0,0,D
2,8635,RSC Anderlecht,Belgium Jupiler League,2008/2009,3,0,W
3,9998,RAEC Mons,Belgium Jupiler League,2008/2009,0,5,L
4,9985,Standard de Liège,Belgium Jupiler League,2008/2009,3,1,W


In [18]:
match_results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51958 entries, 0 to 51957
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   team_id  51958 non-null  int64 
 1   Club     51958 non-null  object
 2   League   51958 non-null  object
 3   Season   51958 non-null  object
 4   GF       51958 non-null  int64 
 5   GA       51958 non-null  int64 
 6   result   51958 non-null  object
dtypes: int64(3), object(4)
memory usage: 2.8+ MB


In [19]:
team_stats = match_results.groupby(["Club","League","Season"])["result"].value_counts().unstack(fill_value=0).fillna(0).astype(int)

In [20]:
team_stats = team_stats.reset_index()

In [21]:
team_stats

result,Club,League,Season,D,L,W
0,1. FC Kaiserslautern,Germany 1. Bundesliga,2010/2011,7,14,13
1,1. FC Kaiserslautern,Germany 1. Bundesliga,2011/2012,11,19,4
2,1. FC Köln,Germany 1. Bundesliga,2008/2009,6,17,11
3,1. FC Köln,Germany 1. Bundesliga,2009/2010,11,14,9
4,1. FC Köln,Germany 1. Bundesliga,2010/2011,5,16,13
...,...,...,...,...,...,...
1473,Śląsk Wrocław,Poland Ekstraklasa,2011/2012,5,8,17
1474,Śląsk Wrocław,Poland Ekstraklasa,2012/2013,8,9,13
1475,Śląsk Wrocław,Poland Ekstraklasa,2013/2014,13,10,7
1476,Śląsk Wrocław,Poland Ekstraklasa,2014/2015,10,8,12


In [22]:
home_goals_for = team_match_home.groupby(["Club","League","Season"])["GF"].sum().reset_index()
home_goals_for = home_goals_for.rename(columns = {
    "GF":"GF_home"})

In [23]:
home_goals_against = team_match_home.groupby(["Club","League","Season"])["GA"].sum().reset_index()
home_goals_against = home_goals_against.rename(columns = {
    "GA":"GA_home"})

In [24]:
away_goals_for = team_match_away.groupby(["Club","League","Season"])["GF"].sum().reset_index()
away_goals_for = away_goals_for.rename(columns = {
    "GF":"GF_away"})

In [25]:
away_goals_against = team_match_away.groupby(["Club","League","Season"])["GA"].sum().reset_index()
away_goals_against = away_goals_against.rename(columns = {
    "GA":"GA_away"})

In [26]:
team_stats = team_stats.merge(
    home_goals_for, 
    on = ["Club","League","Season"], 
    how = "left")

In [27]:
team_stats = team_stats.merge(
    home_goals_against, 
    on = ["Club","League","Season"], 
    how = "left")

In [28]:
team_stats = team_stats.merge(
    away_goals_for, 
    on = ["Club","League","Season"], 
    how = "left")

In [29]:
team_stats = team_stats.merge(
    away_goals_against, 
    on = ["Club","League","Season"], 
    how = "left")

In [30]:
team_stats["GF"] = team_stats['GF_home']+team_stats['GF_away']

In [31]:
team_stats["GA"] = team_stats['GA_home']+team_stats['GA_away']

In [32]:
home_record = team_match_home.groupby(["Club","League","Season"])["result"].value_counts().unstack().fillna(0).astype(int).reset_index()


In [33]:
home_record = home_record.rename(columns = {
    "D":"D_home",
    "W":"W_home",
    "L":"L_home"
})

In [34]:
away_record = team_match_away.groupby(["Club","League","Season"])["result"].value_counts().unstack().fillna(0).astype(int).reset_index()


In [35]:
away_record = away_record.rename(columns = {
    "D":"D_away",
    "W":"W_away",
    "L":"L_away"
})

In [36]:
team_stats = team_stats.merge(
    home_record, 
    on = ["Club","League","Season"], 
    how = "left")

In [37]:
team_stats = team_stats.merge(
    away_record, 
    on = ["Club","League","Season"], 
    how = "left")

In [38]:
team_stats[team_stats["Club"] == "Manchester United"]

,Club,League,Season,D,L,W,GF_home,GA_home,GF_away,GA_away,GF,GA,D_home,L_home,W_home,D_away,L_away,W_away
800,Manchester United,England Premier League,2008/2009,6,4,28,43,13,25,11,68,24,2,1,16,4,3,12
801,Manchester United,England Premier League,2009/2010,4,7,27,52,12,34,16,86,28,1,2,16,3,5,11
802,Manchester United,England Premier League,2010/2011,11,4,23,49,12,29,25,78,37,1,0,18,10,4,5
803,Manchester United,England Premier League,2011/2012,5,5,28,52,19,37,14,89,33,2,2,15,3,3,13
804,Manchester United,England Premier League,2012/2013,5,5,28,45,19,41,24,86,43,0,3,16,5,2,12
805,Manchester United,England Premier League,2013/2014,7,12,19,29,21,35,22,64,43,3,7,9,4,5,10
806,Manchester United,England Premier League,2014/2015,10,8,20,41,15,21,22,62,37,2,3,14,8,5,6
807,Manchester United,England Premier League,2015/2016,9,10,19,27,9,22,26,49,35,5,2,12,4,8,7


In [39]:
team_stats.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1478 entries, 0 to 1477
Data columns (total 18 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Club     1478 non-null   object
 1   League   1478 non-null   object
 2   Season   1478 non-null   object
 3   D        1478 non-null   int64 
 4   L        1478 non-null   int64 
 5   W        1478 non-null   int64 
 6   GF_home  1478 non-null   int64 
 7   GA_home  1478 non-null   int64 
 8   GF_away  1478 non-null   int64 
 9   GA_away  1478 non-null   int64 
 10  GF       1478 non-null   int64 
 11  GA       1478 non-null   int64 
 12  D_home   1478 non-null   int64 
 13  L_home   1478 non-null   int64 
 14  W_home   1478 non-null   int64 
 15  D_away   1478 non-null   int64 
 16  L_away   1478 non-null   int64 
 17  W_away   1478 non-null   int64 
dtypes: int64(15), object(3)
memory usage: 208.0+ KB


In [40]:
team_stats["home_winrate"] = team_stats["W_home"]/(team_stats["W_home"]+team_stats["D_home"]+team_stats["L_home"])

In [41]:
team_stats["away_winrate"] = team_stats["W_away"]/(team_stats["W_home"]+team_stats["D_home"]+team_stats["L_home"])

In [42]:
team_stats.head()

,Club,League,Season,D,L,W,GF_home,GA_home,GF_away,GA_away,GF,GA,D_home,L_home,W_home,D_away,L_away,W_away,home_winrate,away_winrate
0,1. FC Kaiserslautern,Germany 1. Bundesliga,2010/2011,7,14,13,25,19,23,32,48,51,6,5,6,1,9,7,0.352941,0.411765
1,1. FC Kaiserslautern,Germany 1. Bundesliga,2011/2012,11,19,4,12,28,12,26,24,54,5,10,2,6,9,2,0.117647,0.117647
2,1. FC Köln,Germany 1. Bundesliga,2008/2009,6,17,11,14,25,21,25,35,50,5,8,4,1,9,7,0.235294,0.411765
3,1. FC Köln,Germany 1. Bundesliga,2009/2010,11,14,9,18,29,15,13,33,42,6,8,3,5,6,6,0.176471,0.352941
4,1. FC Köln,Germany 1. Bundesliga,2010/2011,5,16,13,30,21,17,41,47,62,2,4,11,3,12,2,0.647059,0.117647


In [43]:
team_stats["home_advantage"] = team_stats["home_winrate"]-team_stats["away_winrate"]

In [44]:
team_stats["home_advantage"].nlargest(15)

1198    0.736842
802     0.684211
779     0.666667
1372    0.647059
1010    0.631579
1195    0.631579
641     0.600000
644     0.600000
683     0.600000
716     0.600000
1130    0.588235
792     0.578947
930     0.578947
519     0.533333
1451    0.533333
Name: home_advantage, dtype: float64

In [45]:
team_stats.sort_values(["home_advantage"],ascending=False)

,Club,League,Season,D,L,W,GF_home,GA_home,GF_away,GA_away,...,GA,D_home,L_home,W_home,D_away,L_away,W_away,home_winrate,away_winrate,home_advantage
1198,Sevilla FC,Spain LIGA BBVA,2015/2016,10,14,14,38,21,13,29,...,50,1,4,14,9,10,0,0.736842,0.000000,0.736842
802,Manchester United,England Premier League,2010/2011,11,4,23,49,12,29,25,...,37,1,0,18,10,4,5,0.947368,0.263158,0.684211
779,Lierse SK,Belgium Jupiler League,2013/2014,0,4,2,5,4,0,8,...,12,0,1,2,0,3,0,0.666667,0.000000,0.666667
1372,VfL Wolfsburg,Germany 1. Bundesliga,2008/2009,6,7,21,51,13,29,28,...,41,1,0,16,5,7,5,0.941176,0.294118,0.647059
1195,Sevilla FC,Spain LIGA BBVA,2012/2013,8,16,14,41,21,17,33,...,54,1,5,13,7,11,1,0.684211,0.052632,0.631579
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1077,Rio Ave FC,Portugal Liga ZON Sagres,2013/2014,8,14,8,10,20,11,15,...,35,5,8,2,3,6,6,0.133333,0.400000,-0.266667
962,Podbeskidzie Bielsko-Biała,Poland Ekstraklasa,2012/2013,8,14,8,21,24,18,19,...,43,6,7,2,2,7,6,0.133333,0.400000,-0.266667
1163,SV Darmstadt 98,Germany 1. Bundesliga,2015/2016,11,14,9,15,29,23,24,...,53,6,9,2,5,5,7,0.117647,0.411765,-0.294118
746,Lech Poznań,Poland Ekstraklasa,2012/2013,4,7,19,24,15,22,7,...,22,3,5,7,1,2,12,0.466667,0.800000,-0.333333


In [46]:
(team_stats
    .sort_values(["home_advantage"], ascending=True)
        .head(25)
    [["Club","Season","home_advantage"]]
)

,Club,Season,home_advantage
695,KV Oostende,2013/2014,-0.666667
746,Lech Poznań,2012/2013,-0.333333
1163,SV Darmstadt 98,2015/2016,-0.294118
962,Podbeskidzie Bielsko-Biała,2012/2013,-0.266667
1077,Rio Ave FC,2013/2014,-0.266667
1232,Sporting Lokeren,2012/2013,-0.266667
754,Lechia Gdańsk,2012/2013,-0.266667
1211,SpVgg Greuther Fürth,2012/2013,-0.235294
1376,VfL Wolfsburg,2012/2013,-0.235294
618,Hibernian,2011/2012,-0.210526


In [47]:
team_stats["home_advantage"].mean()

np.float64(0.17106962197076442)

In [48]:
(team_stats
    .sort_values(["home_advantage"], ascending=True)
        .head(5)
    [["Club","Season","home_advantage"]]
)

,Club,Season,home_advantage
695,KV Oostende,2013/2014,-0.666667
746,Lech Poznań,2012/2013,-0.333333
1163,SV Darmstadt 98,2015/2016,-0.294118
962,Podbeskidzie Bielsko-Biała,2012/2013,-0.266667
1077,Rio Ave FC,2013/2014,-0.266667


In [49]:
team_stats.groupby("Club")["home_advantage"].mean().nsmallest(25)

Club
SV Darmstadt 98                -0.294118
SpVgg Greuther Fürth           -0.235294
Dunfermline Athletic           -0.157895
Tondela                        -0.117647
KV Oostende                    -0.066667
Podbeskidzie Bielsko-Biała     -0.053333
Angers SCO                     -0.052632
Bournemouth                    -0.052632
Córdoba CF                     -0.052632
Hamilton Academical FC         -0.042105
1. FC Kaiserslautern           -0.029412
Moreirense FC                  -0.016993
Blackpool                       0.000000
DSC Arminia Bielefeld           0.000000
FC St. Pauli                    0.000000
Inverness Caledonian Thistle    0.000000
Partick Thistle F.C.            0.000000
RC Recreativo                   0.000000
Reggio Calabria                 0.000000
Watford                         0.000000
Hibernian                       0.008772
FC Vaduz                        0.018519
Aston Villa                     0.026316
FC Zürich                       0.026552
VfL Bochum 

In [50]:
team_stats.groupby("Club")["home_advantage"].mean().nlargest(25)

Club
CD Numancia               0.421053
Catania                   0.415205
Amadora                   0.400000
CD Tenerife               0.368421
Manchester City           0.335526
SC Bastia                 0.328947
RCD Mallorca              0.326316
Jagiellonia Białystok     0.325000
Palermo                   0.323308
Villarreal CF             0.315789
SC Cambuur                0.313725
FC Volendam               0.294118
Tubize                    0.294118
Uniao da Madeira          0.294118
Fulham                    0.289474
Valenciennes FC           0.289474
SC Braga                  0.289216
KV Kortrijk               0.288675
Sevilla FC                0.282895
Genoa                     0.278874
FC Sochaux-Montbéliard    0.271930
Górnik Łęczna             0.266667
Arka Gdynia               0.266667
Sparta Rotterdam          0.264706
Brescia                   0.263158
Name: home_advantage, dtype: float64

In [51]:
team_stats.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1478 entries, 0 to 1477
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Club            1478 non-null   object 
 1   League          1478 non-null   object 
 2   Season          1478 non-null   object 
 3   D               1478 non-null   int64  
 4   L               1478 non-null   int64  
 5   W               1478 non-null   int64  
 6   GF_home         1478 non-null   int64  
 7   GA_home         1478 non-null   int64  
 8   GF_away         1478 non-null   int64  
 9   GA_away         1478 non-null   int64  
 10  GF              1478 non-null   int64  
 11  GA              1478 non-null   int64  
 12  D_home          1478 non-null   int64  
 13  L_home          1478 non-null   int64  
 14  W_home          1478 non-null   int64  
 15  D_away          1478 non-null   int64  
 16  L_away          1478 non-null   int64  
 17  W_away          1478 non-null   i

In [52]:
team_stats.to_csv("team_stats.csv", index = False)

In [53]:
pd.read_csv("team_stats.csv").head()

,Club,League,Season,D,L,W,GF_home,GA_home,GF_away,GA_away,...,GA,D_home,L_home,W_home,D_away,L_away,W_away,home_winrate,away_winrate,home_advantage
0,1. FC Kaiserslautern,Germany 1. Bundesliga,2010/2011,7,14,13,25,19,23,32,...,51,6,5,6,1,9,7,0.352941,0.411765,-0.058824
1,1. FC Kaiserslautern,Germany 1. Bundesliga,2011/2012,11,19,4,12,28,12,26,...,54,5,10,2,6,9,2,0.117647,0.117647,0.000000
2,1. FC Köln,Germany 1. Bundesliga,2008/2009,6,17,11,14,25,21,25,...,50,5,8,4,1,9,7,0.235294,0.411765,-0.176471
3,1. FC Köln,Germany 1. Bundesliga,2009/2010,11,14,9,18,29,15,13,...,42,6,8,3,5,6,6,0.176471,0.352941,-0.176471
4,1. FC Köln,Germany 1. Bundesliga,2010/2011,5,16,13,30,21,17,41,...,62,2,4,11,3,12,2,0.647059,0.117647,0.529412
